---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 🧮 Naive Bayes
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Bayes' theorem gives the exact formula for updating beliefs with evidence. Naive Bayes applies it to classification by assuming predictors are independent given the class — a strong assumption that surprisingly often works in practice.

### Dataset: UCI Adult Income
Same 48,842-record census dataset as the KNN notebook — predict whether income exceeds $50K/year. Using a familiar dataset lets you focus on the method, not the data.

### What you'll learn
- Bayes' theorem step by step: by hand with a single predictor (sex)
- Bayes' theorem with a continuous predictor (age) using Gaussian likelihood
- Full Naive Bayes model across all features
- Why the independence assumption is "naive" but often works
- Gaussian NB vs Bernoulli NB vs Multinomial NB — when to use each

### Time: ~50 minutes

In [ ]:
# WARNING Run this cell first - sets up data and imports
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import GaussianNB, BernoulliNB, CategoricalNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score)
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# Load UCI Adult Income (same as KNN notebook)
try:
    adult = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/adult.csv')
    _src = 'repo CSV'
except Exception:
    raise RuntimeError('Could not load adult.csv -- check internet connection.')

if 'income_bin' not in adult.columns:
    adult['income_bin'] = (adult['income'].str.strip()
                                          .str.replace('.','',regex=False)
                                          .eq('>50K').astype(int))

# Encode sex as binary (Male=1, Female=0)
adult['sex_bin'] = (adult['sex'] == 'Male').astype(int)

print(f'Adult Income ({_src}): {adult.shape}')
print(f'  Income >$50K: {adult["income_bin"].mean():.1%}')
print(f'  Male: {adult["sex_bin"].mean():.1%}  |  Female: {(1-adult["sex_bin"]).mean():.1%}')
print(f'  Age range: {adult["age"].min()} - {adult["age"].max()}, mean={adult["age"].mean():.1f}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('Setup complete')


## 📐 Part 1 — Bayes' Theorem by Hand: Single Predictor (Sex)

**Bayes' theorem:**
```
P(Y=k | X) = P(X | Y=k) · P(Y=k) / P(X)
```

- **P(Y=k)** — prior: overall probability of class k (base rate)
- **P(X | Y=k)** — likelihood: probability of seeing feature value X in class k
- **P(X)** — evidence: overall probability of X (normalising constant)
- **P(Y=k | X)** — posterior: updated probability of class k after seeing X

**Prediction:** assign to the class with the higher posterior.

Let's do this by hand for a single person. We know only their sex. What is the probability they earn >$50K?

In [ ]:
# Bayes' theorem by hand -- single predictor: sex
# Question: given a person is Male, what is P(income >50K)?

n       = len(adult)
n_high  = adult['income_bin'].sum()
n_low   = n - n_high
n_male  = adult['sex_bin'].sum()
n_female = n - n_male

# Count the four combinations
n_male_high   = ((adult['sex_bin']==1) & (adult['income_bin']==1)).sum()
n_male_low    = ((adult['sex_bin']==1) & (adult['income_bin']==0)).sum()
n_female_high = ((adult['sex_bin']==0) & (adult['income_bin']==1)).sum()
n_female_low  = ((adult['sex_bin']==0) & (adult['income_bin']==0)).sum()

print('Contingency table:')
print(f'              Income <=50K   Income >50K   Total')
print(f'  Male:       {n_male_low:>12,}   {n_male_high:>11,}   {n_male:>5,}')
print(f'  Female:     {n_female_low:>12,}   {n_female_high:>11,}   {n_female:>5,}')
print(f'  Total:      {n_low:>12,}   {n_high:>11,}   {n:>5,}')
print()

# Step 1: Prior probabilities
P_high = n_high / n
P_low  = n_low  / n
print(f'Step 1 -- Priors (base rates):')
print(f'  P(income >50K)  = {n_high}/{n} = {P_high:.4f} ({P_high:.1%})')
print(f'  P(income <=50K) = {n_low}/{n}  = {P_low:.4f} ({P_low:.1%})')
print()

# Step 2: Likelihoods P(Male | income class)
P_male_given_high = n_male_high / n_high
P_male_given_low  = n_male_low  / n_low
print(f'Step 2 -- Likelihoods P(Male | income class):')
print(f'  P(Male | >50K)  = {n_male_high}/{n_high} = {P_male_given_high:.4f} ({P_male_given_high:.1%})')
print(f'  P(Male | <=50K) = {n_male_low}/{n_low}  = {P_male_given_low:.4f} ({P_male_given_low:.1%})')
print()

# Step 3: Evidence P(Male)
P_male = n_male / n
print(f'Step 3 -- Evidence:')
print(f'  P(Male) = {n_male}/{n} = {P_male:.4f} ({P_male:.1%})')
print()

# Step 4: Posteriors via Bayes theorem
P_high_given_male = P_male_given_high * P_high / P_male
P_low_given_male  = P_male_given_low  * P_low  / P_male
print(f'Step 4 -- Posteriors (Bayes theorem):')
print(f'  P(>50K  | Male) = P(Male|>50K) x P(>50K) / P(Male)')
print(f'                  = {P_male_given_high:.4f} x {P_high:.4f} / {P_male:.4f}')
print(f'                  = {P_high_given_male:.4f} ({P_high_given_male:.1%})')
print()
print(f'  P(<=50K | Male) = {P_low_given_male:.4f} ({P_low_given_male:.1%})')
print(f'  Sum check:        {P_high_given_male + P_low_given_male:.4f} (must be 1.0)')
print()

# Prediction
pred_male = '>50K' if P_high_given_male > P_low_given_male else '<=50K'
print(f'Prediction for a Male: {pred_male}')
print()

# Verify: same answer via direct proportion
direct = n_male_high / n_male
print(f'Sanity check: n_male_high / n_male = {n_male_high}/{n_male} = {direct:.4f}')
print(f'Matches Bayes result: {abs(direct - P_high_given_male) < 1e-10}')
print()

# Now do the same for Female
P_female = 1 - P_male
P_female_given_high = n_female_high / n_high
P_female_given_low  = n_female_low  / n_low
P_high_given_female = P_female_given_high * P_high / P_female
P_low_given_female  = P_female_given_low  * P_low  / P_female
pred_female = '>50K' if P_high_given_female > P_low_given_female else '<=50K'

print(f'--- Female ---')
print(f'  P(>50K  | Female) = {P_high_given_female:.4f} ({P_high_given_female:.1%})')
print(f'  P(<=50K | Female) = {P_low_given_female:.4f} ({P_low_given_female:.1%})')
print(f'  Prediction: {pred_female}')
print()

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cats = ['Male', 'Female']
p_high_vals  = [P_high_given_male, P_high_given_female]
p_low_vals   = [P_low_given_male,  P_low_given_female]

x = np.arange(2)
axes[0].bar(x, p_high_vals,  color='#e85d20', alpha=0.8, label='>50K')
axes[0].bar(x, p_low_vals,   bottom=p_high_vals, color='#1e5fa8', alpha=0.8, label='<=50K')
axes[0].axhline(P_high, color='grey', lw=1.5, ls='--', label=f'Base rate {P_high:.1%}')
axes[0].set_xticks(x); axes[0].set_xticklabels(cats)
axes[0].set_ylabel('Posterior probability')
axes[0].set_title('P(income | sex) via Bayes theorem')
axes[0].legend(fontsize=9)

# Show lift over base rate
lifts = [P_high_given_male/P_high, P_high_given_female/P_high]
colors = ['#1a7a45' if l > 1 else '#e85d20' for l in lifts]
axes[1].bar(cats, lifts, color=colors, alpha=0.8, edgecolor='white')
axes[1].axhline(1.0, color='grey', lw=1.5, ls='--', label='Base rate (lift=1)')
axes[1].set_ylabel('Lift over base rate')
axes[1].set_title('Lift: how much sex updates the prior')
axes[1].legend(fontsize=9)
for i, (cat, lift) in enumerate(zip(cats, lifts)):
    axes[1].text(i, lift + 0.02, f'{lift:.2f}x', ha='center', fontsize=11)

plt.suptitle('Bayes Theorem by Hand -- Single Predictor: Sex', fontsize=12)
plt.tight_layout()
plt.show()

print('Key insight: sex alone gives a useful signal.')
print(f'  Male:   {P_high_given_male:.1%} chance of >50K  ({P_high_given_male/P_high:.2f}x base rate)')
print(f'  Female: {P_high_given_female:.1%} chance of >50K  ({P_high_given_female/P_high:.2f}x base rate)')
print('But a single predictor is weak. We need more features.')


## 🔬 Part 2 — Bayes by Hand: Continuous Predictor (Age)

For a **continuous** predictor, we can't count exact matches — age=39 may have only a handful of occurrences. Instead, we model the distribution within each class using a **Gaussian (normal) likelihood**:

```
P(age | income=k) = N(age; μₖ, σₖ²)
```

We estimate μₖ and σₖ from the data, then evaluate the Gaussian PDF at the observed age to get the likelihood.

**Prediction for a new person with age = 35:**

```
P(>50K | age=35) ∝ N(35; μ_high, σ_high²) × P(>50K)
P(<=50K| age=35) ∝ N(35; μ_low,  σ_low² ) × P(<=50K)
```

Normalise so they sum to 1.

In [ ]:
# Bayes by hand -- continuous predictor: age
# Gaussian NB assumption: P(age | income=k) ~ Normal(mu_k, sigma_k)

age_high = adult.loc[adult['income_bin']==1, 'age'].values
age_low  = adult.loc[adult['income_bin']==0, 'age'].values

mu_high  = age_high.mean();  sigma_high = age_high.std()
mu_low   = age_low.mean();   sigma_low  = age_low.std()

print('Step 1 -- Class-conditional Gaussian parameters for age:')
print(f'  Income >50K:  mean age = {mu_high:.2f}, std = {sigma_high:.2f}')
print(f'  Income <=50K: mean age = {mu_low:.2f},  std = {sigma_low:.2f}')
print()

# Predict for age = 35
age_query = 35
P_high = adult['income_bin'].mean()
P_low  = 1 - P_high

# Gaussian PDF: P(age | class)
lik_high = stats.norm.pdf(age_query, mu_high, sigma_high)
lik_low  = stats.norm.pdf(age_query, mu_low,  sigma_low)

print(f'Step 2 -- Likelihoods for age = {age_query}:')
print(f'  P(age={age_query} | >50K)  = N({age_query}; {mu_high:.2f}, {sigma_high:.2f}) = {lik_high:.6f}')
print(f'  P(age={age_query} | <=50K) = N({age_query}; {mu_low:.2f},  {sigma_low:.2f})  = {lik_low:.6f}')
print()

# Unnormalised posteriors
unnorm_high = lik_high * P_high
unnorm_low  = lik_low  * P_low
total       = unnorm_high + unnorm_low

post_high = unnorm_high / total
post_low  = unnorm_low  / total

print(f'Step 3 -- Unnormalised posteriors:')
print(f'  P(age={age_query}|>50K)  x P(>50K)  = {lik_high:.6f} x {P_high:.4f} = {unnorm_high:.8f}')
print(f'  P(age={age_query}|<=50K) x P(<=50K) = {lik_low:.6f} x {P_low:.4f} = {unnorm_low:.8f}')
print(f'  Normaliser = {total:.8f}')
print()
print(f'Step 4 -- Posteriors:')
print(f'  P(>50K  | age={age_query}) = {post_high:.4f} ({post_high:.1%})')
print(f'  P(<=50K | age={age_query}) = {post_low:.4f} ({post_low:.1%})')
pred_age35 = '>50K' if post_high > post_low else '<=50K'
print(f'  Prediction: {pred_age35}')
print()

# Show posterior across all ages
age_range = np.linspace(17, 90, 300)
lik_h = stats.norm.pdf(age_range, mu_high, sigma_high)
lik_l = stats.norm.pdf(age_range, mu_low,  sigma_low)
unnorm_h = lik_h * P_high
unnorm_l = lik_l * P_low
post_h_curve = unnorm_h / (unnorm_h + unnorm_l)

# Find crossover age
crossover_idx = np.argmin(np.abs(post_h_curve - 0.5))
crossover_age = age_range[crossover_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: class-conditional Gaussians
axes[0].hist(age_high, bins=40, density=True, alpha=0.4,
             color='#e85d20', label='Income >50K')
axes[0].hist(age_low,  bins=40, density=True, alpha=0.4,
             color='#1e5fa8', label='Income <=50K')
x_g = np.linspace(15, 95, 300)
axes[0].plot(x_g, stats.norm.pdf(x_g, mu_high, sigma_high),
             color='#e85d20', lw=2.5, label=f'>50K fit N({mu_high:.1f},{sigma_high:.1f})')
axes[0].plot(x_g, stats.norm.pdf(x_g, mu_low,  sigma_low),
             color='#1e5fa8', lw=2.5, label=f'<=50K fit N({mu_low:.1f},{sigma_low:.1f})')
axes[0].axvline(age_query, color='#1a7a45', lw=2, ls='--',
                label=f'Query: age={age_query}')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Density')
axes[0].set_title('Gaussian Likelihood P(age | income class)')
axes[0].legend(fontsize=8)

# Right: posterior P(>50K | age)
axes[1].plot(age_range, post_h_curve, color='#e85d20', lw=2.5,
             label='P(>50K | age)')
axes[1].axhline(0.5, color='grey', lw=1, ls='--', label='Decision boundary 0.5')
axes[1].axhline(P_high, color='grey', lw=1, ls=':', label=f'Prior {P_high:.1%}')
axes[1].axvline(age_query, color='#1a7a45', lw=2, ls='--',
                label=f'age={age_query}: P={post_high:.1%}')
axes[1].axvline(crossover_age, color='#1e5fa8', lw=1.5, ls='-.',
                label=f'Crossover age~{crossover_age:.0f}')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('P(income >50K | age)')
axes[1].set_title('Posterior Probability across all ages')
axes[1].legend(fontsize=8)

plt.suptitle('Bayes Theorem by Hand -- Continuous Predictor: Age', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Age alone: predict >50K when age >= ~{crossover_age:.0f}')
print(f'  Higher-income earners tend to be older (mu={mu_high:.1f}) vs lower earners (mu={mu_low:.1f})')
print(f'  But the distributions overlap heavily -- age alone is a weak predictor.')
print(f'  P(>50K | age=35) = {post_high:.1%}  vs prior {P_high:.1%}')


## 📊 Part 3 — Combining Predictors: Naive Bayes

With **two predictors** (sex and age), the exact Bayes update requires the joint distribution P(sex, age | income) — difficult to estimate.

**Naive Bayes** sidesteps this by assuming **conditional independence**:

```
P(sex, age | income=k) = P(sex | income=k) × P(age | income=k)
```

This lets us multiply the individual likelihoods we already computed:

```
P(>50K | sex, age) ∝ P(sex|>50K) × P(age|>50K) × P(>50K)
```

The "naive" assumption is usually wrong — age and sex are correlated. But it often works anyway because what matters is which class gets the higher posterior, not the exact probabilities.

In [ ]:
# Naive Bayes by hand: combine sex + age

# Use the quantities computed in Parts 1 and 2
# Query: Male, age = 35
query_sex = 1  # Male
query_age = 35

# Likelihoods for sex (categorical)
P_male_given_high = n_male_high / n_high
P_male_given_low  = n_male_low  / n_low

# Likelihoods for age (Gaussian)
P_age35_given_high = stats.norm.pdf(query_age, mu_high, sigma_high)
P_age35_given_low  = stats.norm.pdf(query_age, mu_low,  sigma_low)

# Naive Bayes: multiply likelihoods (independence assumption)
score_high = P_male_given_high * P_age35_given_high * P_high
score_low  = P_male_given_low  * P_age35_given_low  * P_low
total_nb   = score_high + score_low

post_nb_high = score_high / total_nb
post_nb_low  = score_low  / total_nb

print('Naive Bayes by hand: Male, age=35')
print('='*55)
print()
print(f'  Prior:         P(>50K) = {P_high:.4f}')
print()
print(f'  Likelihoods:')
print(f'    P(Male   | >50K)  = {P_male_given_high:.4f}')
print(f'    P(age=35 | >50K)  = {P_age35_given_high:.6f}  [Gaussian PDF]')
print(f'    P(Male   | <=50K) = {P_male_given_low:.4f}')
print(f'    P(age=35 | <=50K) = {P_age35_given_low:.6f}  [Gaussian PDF]')
print()
print(f'  Unnormalised scores (likelihood x likelihood x prior):')
print(f'    >50K:  {P_male_given_high:.4f} x {P_age35_given_high:.6f} x {P_high:.4f} = {score_high:.10f}')
print(f'    <=50K: {P_male_given_low:.4f} x {P_age35_given_low:.6f} x {P_low:.4f} = {score_low:.10f}')
print()
print(f'  Posteriors after normalising:')
print(f'    P(>50K  | Male, age=35) = {post_nb_high:.4f} ({post_nb_high:.1%})')
print(f'    P(<=50K | Male, age=35) = {post_nb_low:.4f} ({post_nb_low:.1%})')
print()
pred_nb = '>50K' if post_nb_high > post_nb_low else '<=50K'
print(f'  Prediction: {pred_nb}')
print()

# Compare: sex alone vs age alone vs combined
print('Comparison -- what each predictor contributes:')
print(f'  Sex alone  (Male):     P(>50K) = {P_high_given_male:.1%}')
print(f'  Age alone  (age=35):   P(>50K) = {post_high:.1%}')
print(f'  NB combined:           P(>50K) = {post_nb_high:.1%}')
print(f'  Prior (no info):       P(>50K) = {P_high:.1%}')
print()
print('Each feature updates the prior independently.')
print('Combined, they give a stronger signal than either alone.')


## ✅ Part 4 — Full Naive Bayes Model (sklearn)

Now fit the full Gaussian Naive Bayes model on all numeric features. sklearn does the same calculation we did by hand — just for all features simultaneously.

In [ ]:
# Full Gaussian Naive Bayes on all numeric features
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_curve

num_features = ['age','education_num','hours_per_week',
                'capital_gain','capital_loss','sex_bin']

X = adult[num_features].values.astype(float)
y = adult['income_bin'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

gnb = GaussianNB().fit(X_tr, y_tr)
y_pred = gnb.predict(X_te)
y_prob = gnb.predict_proba(X_te)[:, 1]

auc = roc_auc_score(y_te, y_prob)

print('=== Gaussian Naive Bayes -- Full Model ===')
print(classification_report(y_te, y_pred,
                             target_names=['<=50K', '>50K']))
print(f'AUC-ROC: {auc:.4f}')
print()

# Show learned Gaussian parameters per class
print('Learned class-conditional means (theta_) per feature:')
print(f'{"Feature":15s}  {"Mean (<=50K)":>14}  {"Mean (>50K)":>12}  {"Diff":>8}')
print('-'*58)
for j, fname in enumerate(num_features):
    m0 = gnb.theta_[0][j]
    m1 = gnb.theta_[1][j]
    print(f'{fname:15s}  {m0:>14.3f}  {m1:>12.3f}  {m1-m0:>+8.3f}')
print()

# Verify our hand calculation matches sklearn
query = np.array([[35, 10, 40, 0, 0, 1]])  # Male, age=35, edu=10, 40hrs, no cap
sklearn_prob = gnb.predict_proba(query)[0]
print(f'Sklearn prediction for Male, age=35, edu=10, 40hrs/wk, no capital:')
print(f'  P(<=50K) = {sklearn_prob[0]:.4f}')
print(f'  P(>50K)  = {sklearn_prob[1]:.4f}')
print(f'  (Our 2-feature hand calc gave P(>50K)={post_nb_high:.4f} -- differs')
print(f'   because sklearn uses all 6 features, we used 2)')
print()

# Confusion matrix + ROC
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K'])\
    .plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\nAcc={gnb.score(X_te,y_te):.3f}  AUC={auc:.3f}')

fpr, tpr, _ = roc_curve(y_te, y_prob)
axes[1].plot(fpr, tpr, color='#1e5fa8', lw=2.5, label=f'Naive Bayes (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#1e5fa8')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve -- Gaussian Naive Bayes')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Why "naive" works despite the independence assumption being wrong:')
print('  We only need the right class to have the higher posterior.')
print('  The exact probability values can be wrong -- the ranking just needs to be right.')
print('  This is called the "Naive Bayes optimality" result.')


## 🔍 Part 5 — When to Use Which Naive Bayes

| Variant | Features | Use when |
|---------|----------|----------|
| **Gaussian NB** | Continuous | Features roughly normal within each class |
| **Bernoulli NB** | Binary (0/1) | Binary features: spam words present/absent |
| **Multinomial NB** | Counts | Word counts, term frequencies |
| **Categorical NB** | Categorical | Nominal features with known categories |

**Strengths of Naive Bayes:**
- Extremely fast — O(n·p) to train
- Works well with small n (unlike logistic regression)
- Handles high-dimensional data naturally (text classification)
- Produces calibrated probability estimates (when assumption holds)

**Weaknesses:**
- Independence assumption often violated
- Probabilities can be overconfident (all features reinforce the same direction)
- Gaussian NB fails for skewed/zero-inflated features (log-transform helps)

In [ ]:
# Compare Gaussian NB to Logistic Regression and LDA
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

models = {
    'Naive Bayes (Gaussian)': GaussianNB().fit(X_tr, y_tr),
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_s, y_tr),
    'LDA':                    LinearDiscriminantAnalysis().fit(X_tr, y_tr),
}

print(f'{"Method":28s}  {"Accuracy":>10}  {"AUC-ROC":>9}')
print('-'*52)
for name, model in models.items():
    X_eval = X_te_s if name == 'Logistic Regression' else X_te
    acc = model.score(X_eval, y_te)
    auc_m = roc_auc_score(y_te, model.predict_proba(X_eval)[:,1])
    print(f'{name:28s}  {acc:>10.4f}  {auc_m:>9.4f}')

print()
print('Naive Bayes is competitive despite its simplicity.')
print('For high-income prediction with census data it is a strong baseline.')
print('Logistic regression usually wins when n is large and features are correlated.')
print('Naive Bayes wins when n is small, features are truly independent, or speed matters.')


## 💼 Exercise

**Task 1 — Bayes by hand (new query):** A 55-year-old Female with education_num=14. Using the Gaussian parameters from Part 2, compute P(>50K | age=55) by hand. Then use sklearn to get the full 6-feature prediction. Do they agree in direction?

**Task 2 — Categorical NB:** Encode `workclass`, `education`, and `occupation` as integers and fit `CategoricalNB`. Compare AUC to Gaussian NB.

**Task 3 — Independence check:** For the training set, compute the correlation matrix of the 6 features within each income class. Are they independent? Does the violation appear to hurt Naive Bayes performance?

In [ ]:
# Exercise workspace
# Variables: adult, X_tr, X_te, y_tr, y_te, gnb
#            mu_high, sigma_high, mu_low, sigma_low (from Part 2)
#            P_high, P_low, P_high_given_male (from Part 1)

# Task 1 -- Bayes by hand for age=55
age_query2 = 55
lik_h2 = stats.norm.pdf(age_query2, mu_high, sigma_high)
lik_l2 = stats.norm.pdf(age_query2, mu_low,  sigma_low)
unnorm_h2 = lik_h2 * P_high
unnorm_l2 = lik_l2 * P_low
post_h2 = unnorm_h2 / (unnorm_h2 + unnorm_l2)
print(f'By hand: P(>50K | age=55) = {post_h2:.4f} ({post_h2:.1%})')

# sklearn full model for Female, age=55, edu=14, 40hrs, no capital
query2 = np.array([[55, 14, 40, 0, 0, 0]])  # Female=0
sklearn_prob2 = gnb.predict_proba(query2)[0]
print(f'Sklearn (6 features): P(>50K | Female, age=55, edu=14) = {sklearn_prob2[1]:.4f}')
print()

# Task 3 -- Independence check
print('Feature correlation within each income class:')
for cls, label in [(0,'<=50K'),(1,'>50K')]:
    corr = pd.DataFrame(X_tr[y_tr==cls],
                        columns=num_features).corr().round(2)
    print(f'  {label}:')
    print(corr.to_string())
    print()


In [ ]:
# @title Quiz -- Naive Bayes
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** What does the 'naive' assumption in Naive Bayes mean?
# @markdown **Q2:** In Part 1, why is P(>50K | Male) higher than the base rate P(>50K)?
# @markdown **Q3:** Why do we use the Gaussian PDF for age rather than counting exact matches?
# @markdown **Q4:** The independence assumption is violated here (age and sex are correlated). Why does Naive Bayes still work?
# @markdown **Q5:** Name a situation where Naive Bayes would clearly outperform logistic regression.

q1 = '' # @param {type:'string', placeholder:'your answer'}
q2 = '' # @param {type:'string', placeholder:'your answer'}
q3 = '' # @param {type:'string', placeholder:'your answer'}
q4 = '' # @param {type:'string', placeholder:'your answer'}
q5 = '' # @param {type:'string', placeholder:'your answer'}

answers = {'q1': q1, 'q2': q2, 'q3': q3, 'q4': q4, 'q5': q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f'  {5-len(missing)}/5 answered -- run the submission cell for AI feedback')


In [ ]:
_NB_NAME_ = 'Naive Bayes'
# @title Quiz Submission
# @markdown Click Run, copy the output, paste into Gemini.

GITHUB_USERNAME = '' # @param {type:'string', placeholder:'your GitHub username'}
_NB_TITLE = globals().get('_NB_NAME_', 'this notebook')

if 'answers' not in globals():
    print('Run the quiz cell above first.')
else:
    qa = '\n'.join(f'Q{i}: {str(v).strip() or "(no answer)"}'
                    for i,(_, v) in enumerate(answers.items(), 1))
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip(): print(f'Student: @{GITHUB_USERNAME.strip()}')
    print(); print(qa); print()
    print('For each: CORRECT/PARTIAL/INCORRECT, explanation, ideal answer, Part to revisit.')
    print('End with: Overall grade + short study plan.')


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*